# 강의 03 · 실습 3 — RAG 검색기 구축 · (4) 고난도 I

## 1. 문제상황

- 구름월드 고객센터는 FAQ 외에 「구름월드 이용 규정」 문서를 가지고 있습니다. 조항 8개가 있고 조항마다 문단 하나가 있습니다.
- 손님이 「반려동물 데리고 갈 수 있나요」「분실물은 얼마나 보관하나요」처럼 규정에만 있는 내용을 물으면, 담당자는 규정 문서를 열어 해당 조항을 찾아 읽어 줍니다.
- 규정 문서는 FAQ처럼 행으로 나뉘어 있지 않고 한 파일에 이어져 있어, 그대로 청크(chunk) 하나로 넣으면 검색이 문서 전체를 돌려주어 어느 조항이 답인지 알 수 없습니다.
- 담당자는 규정 문서를 조항 단위로 잘라 검색기에 넣고, 질문에 맞는 조항을 점수와 함께 받기를 원합니다.

## 2. 문제와 목표

- **문제**: 이어 쓴 문서를 청크 하나로 넣으면 검색 단위가 너무 커서 어느 조항이 답인지 가려낼 수 없습니다.
- **목표**: 규정 문서를 조항 단위로 잘라(청킹) 조항 제목을 메타데이터로 실은 청크를 적재하고, 질문에 맞는 조항을 점수와 함께 찾으며, 규정 밖 질문은 임계값으로 컷하는 검색기를 만듭니다.
    - 규정 문서: `gureumworld_rules.md`, `## ` 제목이 붙은 조항 8개. 저장 디렉터리: `chroma_rules`.
    - 조항 제목 메타데이터: `section`(조항 제목)과 `order`(순서).
    - 임계값 1.5와 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」
- **목표 달성 여부의 판정 기준**: 청크 수가 8이고, 「반려동물 데리고 갈 수 있나요?」의 1위 청크가 제5조이며, 「분실물은 얼마나 보관하나요?」의 1위 청크가 제6조이고, 규정 밖 질문이 컷되는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex03_s4_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 조항 단위로 적재합니다.**
    - `gureumworld_rules.md`를 읽어 `## ` 제목 단위로 자릅니다.
    - 청크마다 본문은 제목 줄과 문단을 합친 문자열, 메타데이터는 `{"section": 조항 제목, "order": 순서}`로 하는 `Document`를 만듭니다.
    - 문서 제목(`# ` 줄)은 청크에 넣지 않습니다.
    - 청크 수(8)와 각 청크의 제목·글자 수를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만들고, 문장 하나를 벡터로 바꿔 벡터의 길이를 출력합니다.
3. **저장소를 구축하고 영속합니다.**
    - `persist_directory="chroma_rules"`, `ids`는 `sec-<순서>`로 합니다.
    - 항목 수를 출력합니다.
4. **점수와 함께 검색합니다.**
    - 「반려동물 데리고 갈 수 있나요?」와 「분실물은 얼마나 보관하나요?」를 `k=2`로 검색해 청크의 제목과 점수를 출력합니다.
5. **임계값으로 컷합니다.**
    - `THRESHOLD = 1.5`와 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`를 두고, 1위 점수가 임계값 이하이면 조항 제목과 본문 앞 80자를, 넘으면 `NO_EVIDENCE`를 돌려주는 함수 `answer_or_cut`을 만듭니다.
    - 규정 안 질문(「주차는 얼마까지 무료인가요?」)과 규정 밖 질문(「파이썬 리스트 정렬은 어떻게 하나요?」)을 넣어 출력합니다.
    - 판정은 「통과」 또는 「컷」으로 표시합니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단

다섯 단계는 FAQ 검색기와 같고 ① 적재부만 교체됩니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `text.split("\n## ")`, `Document(page_content=..., metadata={"section": ...})` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(chunks, emb, persist_directory="chroma_rules", ids=...)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=2)` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `if s <= THRESHOLD` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다. 임베딩 모델도 OpenAI API를 쓰므로 같은 키가 필요합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import csv
import os

from dotenv import load_dotenv, find_dotenv

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- 청킹은 검색 단위로 자르는 일입니다. 이 문서는 조항마다 `## ` 제목이 있으므로 제목을 경계로 자릅니다.
- 제목을 메타데이터에 실어 두면 검색 결과에서 어느 조항인지 바로 읽을 수 있습니다.

In [ ]:
# 여기에 단계 ①(마크다운 조항 청킹)을 작성합니다.

### 단계 ② — 임베딩 준비 (요구사항 2)

- 임베딩은 문장을 숫자 벡터로 바꾸는 모델입니다. 의미가 가까운 문장은 벡터도 가깝습니다.
- 문서를 넣을 때와 질문을 넣을 때 같은 임베딩을 써야 같은 좌표계에서 거리를 잴 수 있습니다.

In [ ]:
# 여기에 단계 ②(임베딩 준비)를 작성합니다.

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- `Chroma.from_documents`가 문서마다 임베딩을 계산해 저장소에 넣습니다. `persist_directory`를 주면 디렉터리에 남아 프로그램이 끝나도 유지됩니다.
- 문서 id를 조항 순서로 주면 같은 셀을 다시 실행해도 같은 id에 덮어써 항목이 늘지 않습니다.

In [ ]:
# 여기에 단계 ③(구축·영속)을 작성합니다.

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- 청크의 제목을 메타데이터에서 읽어 어느 조항이 답인지 확인합니다.

In [ ]:
# 여기에 단계 ④(질문 두 개 검색 k=2)를 작성합니다.

### 단계 ⑤ — 임계값 컷 (요구사항 5)

- 임계값은 문서 안 질문의 점수 분포와 문서 밖 질문의 점수 분포 사이에 긋는 선입니다. 여기서는 1.5를 씁니다.
- 컷된 질문을 보낼 곳을 정해야 설계가 닫힙니다. 이 실습에서는 고정 안내 문장으로 보냅니다.

In [ ]:
# 여기에 단계 ⑤(임계값 컷 함수와 두 질문 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 청크 수가 8이고, 제1조부터 제8조까지 제목이 순서대로 찍히며, 단계 ③의 항목 수도 8입니다.
2. 단계 ④에서 반려동물 질문의 1위가 「제5조 반려동물」, 분실물 질문의 1위가 「제6조 분실물」입니다.
3. 단계 ⑤에서 주차 질문은 「통과」와 「제4조 주차 — …」가, 파이썬 질문은 「컷」과 고정 안내 문장이 찍힙니다.

세 가지가 모두 확인되면 완성입니다.